# 🐦‍⬛ Raven Nano — Training Pipeline

10M parameter transformer LLM trained from scratch.

**Execution order:**
1. Mount Drive → Install deps
2. Clone repo → Upload corpus
3. Train tokenizer → Prepare data
4. Train model → Test inference

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/raven_nano/checkpoints', exist_ok=True)
print('Drive mounted. Checkpoint directory ready.')

## 2. Install Dependencies

In [ ]:
!pip install tokenizers --quiet
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 3. Clone Repo

In [ ]:
# Replace with your actual repo URL
!git clone https://github.com/itslokeshx/Raven-Nano.git
%cd Raven-Nano

## 4. Upload Corpus

In [ ]:
from google.colab import files
import os

os.makedirs('data', exist_ok=True)

# Option A: Upload from your machine
uploaded = files.upload()
for filename, data in uploaded.items():
    with open(f'data/{filename}', 'wb') as f:
        f.write(data)
    print(f'Saved: data/{filename}')

# Option B: Copy from Drive (uncomment if corpus is already on Drive)
# !cp '/content/drive/MyDrive/raven_nano/corpus.txt' data/corpus.txt

## 5. Train Tokenizer

In [ ]:
!python tokenizer_train.py

## 6. Prepare Data

In [ ]:
!python data_prep.py

## 7. Train (Fresh Start)

In [ ]:
!python train.py

## 7b. Resume Training (if disconnected)

In [ ]:
!python train.py --resume

## 8. Check Training Progress

In [ ]:
import torch
ckpt = torch.load('checkpoints/raven_nano_best.pt', map_location='cpu')
print(f'Best checkpoint:')
print(f'  Step     : {ckpt["step"]}')
print(f'  Val loss : {ckpt["val_loss"]:.4f}')

## 9. Test Inference

In [ ]:
import sys
sys.path.insert(0, '.')

from inference import load_model_for_inference, sample
from tokenizers import Tokenizer
from config import cfg

tok   = Tokenizer.from_file(cfg.TOKENIZER_PATH)
model = load_model_for_inference()

test_inputs = [
    'hi',
    "i can't study tonight",
    'gonna hit the gym',
    "i'm a failure",
    'what is 47 times 83',
    'good morning',
    "i'll start tomorrow",
]

print('=' * 50)
for inp in test_inputs:
    response = sample(model, tok, inp)
    print(f'User  : {inp}')
    print(f'Raven : {response}')
    print()